# IMDb Movie Review Sentiment Analysis

A reproducible workflow for exploring 50,000 IMDb reviews, preparing text, and comparing classical machine-learning baselines with LSTM and BiLSTM neural networks. The final section evaluates all models on the same stratified test set and demonstrates inference with the saved BiLSTM model.

**Target:** positive or negative sentiment  
**Evaluation:** accuracy, precision, recall, F1 score, ROC-AUC, and confusion matrices


In [ ]:
import re
import string
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from wordcloud import WordCloud
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42


## 1. Load the Dataset

Place `IMDB Dataset.csv` beside this notebook, or update `DATA_PATH` below.


In [ ]:
DATA_PATH = Path("IMDB Dataset.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. Place IMDB Dataset.csv beside the notebook."
    )

df = pd.read_csv(DATA_PATH)
required_columns = {"review", "sentiment"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")
print(f"Loaded {len(df):,} reviews.")
df.head()


## 2. Exploratory Data Analysis


In [ ]:
df.head()


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df['sentiment'].value_counts()

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(data=df, x='sentiment')

plt.title('Distribution of Sentiments')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')

plt.show()

In [ ]:
df["review_length"] = df["review"].str.len()


In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    data=df,
    x='review_length',
    hue='sentiment',
    bins=50,
    kde=True
)

plt.title('Distribution of Review Lengths by Sentiment')
plt.xlabel('Review Length')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df.groupby('sentiment')['review_length'].mean()

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(
    data=df,
    x='sentiment',
    y='review_length',
    estimator='mean'
)

plt.title('Average Review Length by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Average Characters')

plt.show()

In [ ]:
positive_text = " ".join(
    df[df['sentiment'] == 'positive']['review']
)

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color='white',
    max_words=100
).generate(positive_text)

plt.figure(figsize=(14,7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words in Positive Reviews')
plt.show()

In [ ]:
negative_text = " ".join(
    df[df['sentiment'] == 'negative']['review']
)

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color='white',
    max_words=100
).generate(negative_text)

plt.figure(figsize=(14,7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words in Negative Reviews')
plt.show()

## 3. Text Cleaning and Preparation

We remove exact duplicate rows, normalize review text, and retain negation terms because they can change sentiment.


### 3.1 Remove duplicate reviews


In [ ]:
print(f"Duplicate rows before removal: {df.duplicated().sum():,}")


In [ ]:
df = df.drop_duplicates()

In [ ]:
print(f"Duplicate rows after removal: {df.duplicated().sum():,}")


### 3.2 Clean review text


In [ ]:
# Keep negation words because they are important for sentiment.
negation_words = {"no", "not", "nor", "never"}
stop_words = set(__import__("nltk").corpus.stopwords.words("english")) - negation_words

def clean_text(text):
    """Normalize an IMDb review and remove common English stopwords."""
    text = re.sub(r"<.*?>", " ", str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " ", text.lower())
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"[^a-z\s]", " ", text)
    words = (word for word in text.split() if word not in stop_words)
    return re.sub(r"\s+", " ", " ".join(words)).strip()


In [ ]:
df["clean_review"] = df["review"].map(clean_text)
df[["review", "clean_review"]].head()


In [ ]:
df[['review', 'clean_review']].head()

In [ ]:
df["original_length"] = df["review"].str.len()
df["clean_length"] = df["clean_review"].str.len()


In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    data=df,
    x='clean_length',
    hue='sentiment',
    bins=50,
    kde=True
)

plt.title('Distribution of Cleaned Review Lengths')
plt.xlabel('Review Length After Preprocessing')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df['original_length'] = df['review'].apply(len)
df['clean_length'] = df['clean_review'].apply(len)

In [ ]:

# Calculate average lengths
original_avg = df['original_length'].mean()
clean_avg = df['clean_length'].mean()

# Create data for plotting
length_data = pd.DataFrame({
    'Stage': ['Before Cleaning', 'After Cleaning'],
    'Average Length': [original_avg, clean_avg]
})

# Plot
plt.figure(figsize=(9, 6))

sns.barplot(
    data=length_data,
    x='Stage',
    y='Average Length'
)

plt.title('Average Review Length Before vs After Preprocessing',
          fontsize=15, fontweight='bold')

plt.xlabel('Preprocessing Stage')
plt.ylabel('Average Number of Characters')

plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

# Get all words from positive reviews
positive_words = ' '.join(
    df[df['sentiment'] == 'positive']['clean_review']
).split()

# Count word frequency
positive_word_counts = Counter(positive_words)

# Get top 20 words
top_20_positive = positive_word_counts.most_common(20)

# Convert to DataFrame
top_positive_df = pd.DataFrame(
    top_20_positive,
    columns=['Word', 'Frequency']
)

# Plot
plt.figure(figsize=(12, 7))

sns.barplot(
    data=top_positive_df,
    x='Frequency',
    y='Word'
)

plt.title(
    'Top 20 Most Frequent Words in Positive Reviews',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Frequency')
plt.ylabel('Word')

plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

# Negative reviews only
negative_reviews = df[df['sentiment'] == 'negative']['clean_review']

# Combine all words
negative_words = ' '.join(negative_reviews).split()

# Count words
word_counts = Counter(negative_words)

# Top 20 words
top_20_negative = word_counts.most_common(20)

# Create DataFrame
top_negative_df = pd.DataFrame(
    top_20_negative,
    columns=['Word', 'Frequency']
)

# Plot
plt.figure(figsize=(12, 7))

sns.barplot(
    data=top_negative_df,
    x='Frequency',
    y='Word'
)

plt.title(
    'Top 20 Most Frequent Words in Negative Reviews',
    fontsize=16,
    fontweight='bold'
)

plt.xlabel('Frequency')
plt.ylabel('Word')

plt.tight_layout()
plt.show()

## 4. Encode Labels and Split the Data


In [ ]:
encoder = LabelEncoder()
df["sentiment_encoded"] = encoder.fit_transform(df["sentiment"])
print("Label mapping:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))


### Stratified train/test split


In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_review']
y = df['sentiment_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training size:", len(X_train))
print("Testing size:", len(X_test))

## 5. Classical Baselines: BoW and TF-IDF

Logistic Regression with Bag-of-Words and TF-IDF provides interpretable reference points. Vectorizers are fitted on training data only.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Bag-of-Words counts and TF-IDF weights, using unigrams and bigrams.
bow_vectorizer = CountVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
tfidf_vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

bow_model = LogisticRegression(max_iter=1000, solver="liblinear")
tfidf_model = LogisticRegression(max_iter=1000, solver="liblinear")

bow_model.fit(X_train_bow, y_train)
tfidf_model.fit(X_train_tfidf, y_train)

y_pred_bow = bow_model.predict(X_test_bow)
y_pred_tfidf = tfidf_model.predict(X_test_tfidf)
y_prob_bow = bow_model.predict_proba(X_test_bow)[:, 1]
y_prob_tfidf = tfidf_model.predict_proba(X_test_tfidf)[:, 1]

baseline_results = {
    "BoW + Logistic Regression": {
        "accuracy": accuracy_score(y_test, y_pred_bow),
        "precision": precision_score(y_test, y_pred_bow, zero_division=0),
        "recall": recall_score(y_test, y_pred_bow, zero_division=0),
        "f1": f1_score(y_test, y_pred_bow, zero_division=0),
    },
    "TF-IDF + Logistic Regression": {
        "accuracy": accuracy_score(y_test, y_pred_tfidf),
        "precision": precision_score(y_test, y_pred_tfidf, zero_division=0),
        "recall": recall_score(y_test, y_pred_tfidf, zero_division=0),
        "f1": f1_score(y_test, y_pred_tfidf, zero_division=0),
    },
}

baseline_scores = pd.DataFrame(baseline_results).T
baseline_scores = baseline_scores.rename(columns={"f1": "f1_score"})
display(baseline_scores.round(4))

print("BoW classification report:")
print(classification_report(y_test, y_pred_bow, target_names=["Negative", "Positive"], zero_division=0))
print("TF-IDF classification report:")
print(classification_report(y_test, y_pred_tfidf, target_names=["Negative", "Positive"], zero_division=0))

## 6. Prepare Sequences for Neural Models

The tokenizer is fitted on the training split only. Both recurrent models use the same vocabulary limit and sequence length.


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 10000
MAX_LENGTH = 200

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)


In [ ]:
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LENGTH, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LENGTH, padding="post", truncating="post")
print("Training sequence shape:", X_train_pad.shape)
print("Test sequence shape:", X_test_pad.shape)


## 7. Train and Evaluate Neural Models


### 7.1 LSTM


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout

vocab_size = VOCAB_SIZE
max_length = MAX_LENGTH

lstm_model = Sequential([
    Input(shape=(max_length,)),
    
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    
    LSTM(128),
    
    Dropout(0.5),
    
    Dense(64, activation='relu'),
    
    Dropout(0.3),
    
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

def make_early_stopping():
    return EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)


In [ ]:
history_lstm = lstm_model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[make_early_stopping()],
)

In [ ]:
test_loss, test_accuracy = lstm_model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Predictions
y_pred_prob = lstm_model.predict(X_test_pad, verbose=0)

# Convert probabilities to 0 or 1
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-Score :", f1)

In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_
))

In [ ]:
### 7.2 Bidirectional LSTM (BiLSTM)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional,
    LSTM, Dense, Dropout
)

vocab_size = VOCAB_SIZE
max_length = MAX_LENGTH

bilstm_model = Sequential([
    Input(shape=(max_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    Bidirectional(
        LSTM(128)
    ),

    Dropout(0.5),

    Dense(64, activation='relu'),

    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

bilstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

bilstm_model.summary()

In [ ]:
history_bilstm = bilstm_model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[make_early_stopping()]
)

In [ ]:
test_loss_bilstm, test_accuracy_bilstm = bilstm_model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print("BiLSTM Test Loss:", test_loss_bilstm)
print("BiLSTM Test Accuracy:", test_accuracy_bilstm)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Predictions
y_pred_prob_bilstm = bilstm_model.predict(X_test_pad, verbose=0)

# Convert probabilities to 0 or 1
y_pred_bilstm = (
    y_pred_prob_bilstm >= 0.5
).astype(int).flatten()

# Metrics
accuracy_bilstm = accuracy_score(y_test, y_pred_bilstm)
precision_bilstm = precision_score(y_test, y_pred_bilstm)
recall_bilstm = recall_score(y_test, y_pred_bilstm)
f1_bilstm = f1_score(y_test, y_pred_bilstm)

print("Accuracy :", accuracy_bilstm)
print("Precision:", precision_bilstm)
print("Recall   :", recall_bilstm)
print("F1-Score :", f1_bilstm)

In [ ]:
print(classification_report(
    y_test,
    y_pred_bilstm,
    target_names=encoder.classes_
))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_bilstm)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)

plt.title('BiLSTM Confusion Matrix', fontsize=15, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

plt.tight_layout()
plt.show()

## 8. Compare Model Performance

The plots compare learning curves and test-set metrics for all four models.


## 9. Save the Model and Tokenizer

Save artifacts beside the notebook so they can be loaded by the inference example below.


In [ ]:
bilstm_model.save("bilstm_model.keras")
lstm_model.save("lstm_model.keras")


## 10. Example Prediction

This example applies the same text cleaning and tokenization steps used during training.


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

review = "I love this movie. It is amazing and entertaining."
cleaned_review = clean_text(review)
sequence = tokenizer.texts_to_sequences([cleaned_review])
padded = pad_sequences(sequence, maxlen=MAX_LENGTH, padding="post", truncating="post")
positive_score = float(bilstm_model.predict(padded, verbose=0)[0][0])
prediction = "positive" if positive_score >= 0.5 else "negative"
print("Prediction:", prediction)
print(f"Positive probability: {positive_score:.2%}")


In [ ]:
import pickle

with open("tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)
print("Saved both neural models and tokenizer.")
